# Factor Analysis Model

## Learning Objectives
- Understand why MLE fails when $n \gg m$ (singular covariance)
- Know the two restricted covariance approaches (diagonal $\Sigma$, isotropic $\sigma^2 I$) and their limitations
- Define the **Factor Analysis** generative model: $z \sim \mathcal{N}(0, I)$, $x|z \sim \mathcal{N}(\mu + \Lambda z, \Psi)$
- Derive the joint distribution of $(z, x)$ and the marginal $x \sim \mathcal{N}(\mu, \Lambda\Lambda^T + \Psi)$
- Understand $\Lambda$ as a **factor loading matrix** mapping low-dimensional $z$ to high-dimensional $x$

## Problem Statement

**Challenge:** With $m$ data points in $\mathbb{R}^n$, the sample covariance $\hat{\Sigma} = \frac{1}{m}\sum_i(x^{(i)}-\mu)(x^{(i)}-\mu)^T$ is singular when $m < n+1$ — the data spans only a low-dimensional subspace.

**Restricted approaches** (fit with small $m$, but lose expressivity):
- **Diagonal $\Sigma$:** $\hat{\Sigma}_{jj} = \frac{1}{m}\sum_i(x^{(i)}_j - \mu_j)^2$ — assumes feature independence
- **Isotropic:** $\Sigma = \sigma^2 I$, $\hat{\sigma}^2 = \frac{1}{mn}\sum_j\sum_i(x^{(i)}_j - \mu_j)^2$ — all variances equal

**Factor Analysis** captures covariance through a latent $k$-dimensional factor $z$ ($k \ll n$):

**Generative model:**
1. Draw latent factor: $z \sim \mathcal{N}(0, I)$, $\quad z \in \mathbb{R}^k$
2. Draw observation: $x \mid z \sim \mathcal{N}(\mu + \Lambda z,\; \Psi)$, $\quad x \in \mathbb{R}^n$

**Parameters:** $\mu \in \mathbb{R}^n$, loading matrix $\Lambda \in \mathbb{R}^{n \times k}$, diagonal noise $\Psi \in \mathbb{R}^{n \times n}$.

**Equivalently:** $x = \mu + \Lambda z + \varepsilon$, with $\varepsilon \sim \mathcal{N}(0, \Psi)$, $z \perp \varepsilon$.

**Joint distribution:**
$$\begin{bmatrix} z \\ x \end{bmatrix} \sim \mathcal{N}\!\left(
\begin{bmatrix} 0 \\ \mu \end{bmatrix},
\begin{bmatrix} I & \Lambda^T \\ \Lambda & \Lambda\Lambda^T + \Psi \end{bmatrix}\right)$$

**Marginal:** $x \sim \mathcal{N}(\mu,\; \Lambda\Lambda^T + \Psi)$

The marginal covariance $\Lambda\Lambda^T + \Psi$ has a **low-rank + diagonal** structure — rich enough to capture correlations with only $nk + n$ parameters instead of $n^2/2$.

## 1. The $n \gg m$ Problem: Why Full Covariance Fails

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Show rank deficiency of sample covariance when m < n
n = 50    # dimension
m_vals = [10, 30, 51, 100, 500]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: rank of sample covariance vs m
ranks, min_eigs = [], []
for m in m_vals:
    X = np.random.randn(m, n)
    Sig = (X - X.mean(axis=0)).T @ (X - X.mean(axis=0)) / m
    ranks.append(np.linalg.matrix_rank(Sig))
    eigs = np.linalg.eigvalsh(Sig)
    min_eigs.append(eigs.min())

ax = axes[0]
ax.bar(range(len(m_vals)), ranks, color=['#d6604d' if r < n else '#4dac26' for r in ranks],
       edgecolor='k')
ax.axhline(n, color='k', ls='--', lw=2, label=f'Full rank = {n}')
ax.set_xticks(range(len(m_vals)))
ax.set_xticklabels([f'm={m}' for m in m_vals])
ax.set_ylabel('Rank of $\\hat{\\Sigma}$')
ax.set_title(f'Sample covariance rank (n={n})\nRed = singular, green = full rank')
ax.legend(fontsize=9)

# Right: parameter count comparison
ax = axes[1]
n_vals = np.arange(5, 200, 5)
k_fa   = 10
params_full = n_vals * (n_vals + 1) / 2
params_diag = n_vals
params_iso  = np.ones_like(n_vals)
params_fa   = n_vals * k_fa + n_vals   # nk + n

ax.semilogy(n_vals, params_full, 'b-', lw=2.5, label='Full $\\Sigma$: $n(n+1)/2$')
ax.semilogy(n_vals, params_fa,   'r-', lw=2.5, label=f'Factor Analysis (k={k_fa}): $nk+n$')
ax.semilogy(n_vals, params_diag, 'g-', lw=2.5, label='Diagonal $\\Sigma$: $n$')
ax.semilogy(n_vals, params_iso,  'k--',lw=2,   label='Isotropic: $1$')
ax.set_xlabel('Dimension $n$')
ax.set_ylabel('Number of parameters (log scale)')
ax.set_title('Parameter count: FA is between diagonal and full')
ax.legend(fontsize=9)

fig.suptitle('The $n \\gg m$ Challenge and Covariance Parameterisation Tradeoffs',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('When m < n: sample covariance is rank-deficient (singular), MLE undefined')
print(f'FA with k={k_fa}, n=100: {100*k_fa + 100} params vs {100*101//2} for full Σ')

## 2. Factor Analysis Generative Story

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

# 2D observed, 1D latent — easy to visualise
n, k = 2, 1
mu_fa  = np.array([1.0, 2.0])
Lambda = np.array([[1.5], [1.0]])   # loading matrix (2x1)
Psi    = np.diag([0.3, 0.5])        # diagonal noise

m = 300
z = np.random.randn(m, k)
eps = np.random.multivariate_normal(np.zeros(n), Psi, m)
X   = mu_fa + z @ Lambda.T + eps   # x = mu + Lambda*z + epsilon

# Theoretical marginal covariance
Sig_x = Lambda @ Lambda.T + Psi

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: latent z distribution
ax = axes[0]
ax.hist(z[:, 0], bins=40, density=True, color='steelblue', alpha=0.7)
from scipy.stats import norm
t = np.linspace(-4, 4, 200)
ax.plot(t, norm.pdf(t), 'k-', lw=2.5, label='$\\mathcal{N}(0,1)$')
ax.set_xlabel('$z$'); ax.set_ylabel('Density')
ax.set_title('Step 1: Draw $z \\sim \\mathcal{N}(0, I)$\n(1D latent factor)')
ax.legend(fontsize=9)

# Middle: show how Lambda*z maps to 2D space
ax = axes[1]
lz = z @ Lambda.T  # (m, 2)
ax.scatter(lz[:, 0], lz[:, 1], s=20, alpha=0.4, c='steelblue', label='$\\Lambda z$ (noiseless)')
ax.scatter(X[:, 0], X[:, 1],   s=20, alpha=0.4, c='#d6604d',   label='$x = \\mu + \\Lambda z + \\epsilon$')

# Draw the direction of Lambda
lam_dir = Lambda[:, 0] / np.linalg.norm(Lambda[:, 0])
ax.annotate('', xy=mu_fa + 2*lam_dir, xytext=mu_fa,
            arrowprops=dict(arrowstyle='->', color='k', lw=2.5, mutation_scale=18))
ax.text(mu_fa[0]+2*lam_dir[0]+0.1, mu_fa[1]+2*lam_dir[1], '$\\Lambda$', fontsize=13)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Step 2: Project to 2D then add noise\n$x = \\mu + \\Lambda z + \\epsilon$')
ax.legend(fontsize=9)

# Right: marginal x and theoretical ellipse
ax = axes[2]
ax.scatter(X[:, 0], X[:, 1], s=15, alpha=0.3, c='steelblue')

# Draw covariance ellipse
from matplotlib.patches import Ellipse
vals, vecs = np.linalg.eigh(Sig_x)
order = vals.argsort()[::-1]
vals, vecs = vals[order], vecs[:, order]
angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
for n_std, alpha in [(1, 0.6), (2, 0.3)]:
    ell = Ellipse(xy=mu_fa, width=2*n_std*np.sqrt(vals[0]),
                  height=2*n_std*np.sqrt(vals[1]), angle=angle,
                  fc='none', ec='red', lw=2, alpha=alpha)
    ax.add_patch(ell)
ax.scatter(*mu_fa, s=150, c='red', zorder=5, marker='X')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Marginal $x \\sim \\mathcal{N}(\\mu, \\Lambda\\Lambda^T + \\Psi)$\n'
             'Red ellipses = theoretical 1σ, 2σ')

print('Theoretical Σ_x = ΛΛᵀ + Ψ =')
print(Sig_x.round(3))
print('\nEmpirical Σ_x =')
print(np.cov(X.T).round(3))

fig.suptitle('Factor Analysis Generative Story: $z \\to x = \\mu + \\Lambda z + \\epsilon$',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Deriving the Joint and Marginal Distributions

Since $x = \mu + \Lambda z + \varepsilon$ with $z \perp \varepsilon$:

$$\mathrm{Cov}(z, x) = E[z\,(\Lambda z + \varepsilon)^T] = E[zz^T]\Lambda^T = \Lambda^T$$

$$\mathrm{Cov}(x) = E[(\Lambda z + \varepsilon)(\Lambda z + \varepsilon)^T] = \Lambda E[zz^T]\Lambda^T + E[\varepsilon\varepsilon^T] = \Lambda\Lambda^T + \Psi$$

The **low-rank + diagonal** structure $\Lambda\Lambda^T + \Psi$ is the key: $\Lambda\Lambda^T$ captures correlations between features through the shared latent factors, while $\Psi$ accounts for feature-specific noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualise the low-rank + diagonal structure of the marginal covariance
n_vis, k_vis = 8, 2
rng = np.random.default_rng(5)
Lambda_vis = rng.standard_normal((n_vis, k_vis))
Psi_vis    = np.diag(rng.uniform(0.1, 0.5, n_vis))

LLT = Lambda_vis @ Lambda_vis.T
Sig = LLT + Psi_vis

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
cmap = 'RdBu_r'

for ax, mat, title in [
    (axes[0], LLT,     f'$\\Lambda\\Lambda^T$ (rank {k_vis})\nLow-rank: captures factor correlations'),
    (axes[1], Psi_vis, '$\\Psi$ (diagonal)\nFeature-specific noise'),
    (axes[2], Sig,     '$\\Lambda\\Lambda^T + \\Psi$\nMarginal covariance of $x$'),
    (axes[3], np.diag(np.linalg.eigvalsh(Sig)[::-1]), 'Eigenvalues of $\\Sigma_x$\nFirst k are large (from $\\Lambda$)'),
]:
    vmax = np.abs(mat).max()
    im = ax.imshow(mat, cmap=cmap, vmin=-vmax, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=9)

fig.suptitle('Low-Rank + Diagonal Structure of Factor Analysis Covariance',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

eigs = np.linalg.eigvalsh(Sig)[::-1]
print(f'Eigenvalues of Σ_x: {eigs.round(3)}')
print(f'Large eigenvalues (from ΛΛᵀ): {eigs[:k_vis].round(3)}')
print(f'Small eigenvalues (from Ψ):   {eigs[k_vis:].round(3)}')

## 4. Factor Analysis vs. Restricted Gaussian vs. PCA

| Method | Covariance model | Captures correlations? | Handles $n \gg m$? | Noise model |
|---|---|---|---|---|
| Full MLE | $\hat{\Sigma}$ (free) | Yes | No (singular) | Implicit |
| Diagonal $\Sigma$ | $\text{diag}(\sigma_j^2)$ | No | Yes | Heteroscedastic |
| Isotropic | $\sigma^2 I$ | No | Yes | Homoscedastic |
| **Factor Analysis** | $\Lambda\Lambda^T + \Psi$ | Yes (via $\Lambda$) | Yes ($nk + n$ params) | Heteroscedastic diagonal |
| PCA | $\Lambda\Lambda^T$ (implied) | Yes | Yes | Implicit (no noise term) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Generate data from a true FA model and compare covariance fits
n_true, k_true, m_data = 10, 2, 30   # n > m — the hard regime
Lambda_true = np.random.randn(n_true, k_true)
Psi_true    = np.diag(np.random.uniform(0.2, 1.0, n_true))
mu_true     = np.zeros(n_true)
Sig_true    = Lambda_true @ Lambda_true.T + Psi_true

z_data  = np.random.randn(m_data, k_true)
eps_data = np.random.multivariate_normal(np.zeros(n_true), Psi_true, m_data)
X_data  = z_data @ Lambda_true.T + eps_data

# Covariance estimates
Xc = X_data - X_data.mean(axis=0)
Sig_full = Xc.T @ Xc / m_data                        # rank-deficient
Sig_diag = np.diag(np.var(X_data, axis=0))            # diagonal
Sig_iso  = np.var(X_data) * np.eye(n_true)            # isotropic

# Factor Analysis fit via sklearn
from sklearn.decomposition import FactorAnalysis
fa = FactorAnalysis(n_components=k_true, random_state=0)
fa.fit(X_data)
Lambda_hat = fa.components_.T      # sklearn stores as (k, n)
Psi_hat    = np.diag(fa.noise_variance_)
Sig_fa     = Lambda_hat @ Lambda_hat.T + Psi_hat

# Compare Frobenius errors
models = [('True $\\Sigma$', Sig_true), ('Full $\\hat{\\Sigma}$', Sig_full),
          ('Diagonal', Sig_diag), ('Isotropic', Sig_iso), ('FA fit', Sig_fa)]
errors = [(name, np.linalg.norm(S - Sig_true, 'fro')) for name, S in models]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
names  = [e[0] for e in errors]
errs   = [e[1] for e in errors]
colors_bar = ['green', '#d6604d', 'gray', 'gray', '#2166ac']
ax.bar(range(len(errors)), errs, color=colors_bar, edgecolor='k', alpha=0.8)
ax.set_xticks(range(len(errors)))
ax.set_xticklabels(names, rotation=15, fontsize=9)
ax.set_ylabel('Frobenius error $\\|\\hat{\\Sigma} - \\Sigma_{\\mathrm{true}}\\|_F$')
ax.set_title(f'Covariance estimation error (n={n_true}, m={m_data})')
ax.text(0.5, 0.95, f'n={n_true} > m={m_data}: FA fits better than full or naive',
        transform=ax.transAxes, ha='center', va='top', fontsize=9,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

ax = axes[1]
vmax = np.abs(Sig_true).max()
ax.imshow(np.abs(Sig_true - Sig_fa), vmin=0, vmax=vmax*0.5, cmap='Reds')
ax.set_title('|True Σ − FA fit|\n(residual error matrix)')
ax.set_xlabel('Feature index'); ax.set_ylabel('Feature index')

fig.suptitle('Factor Analysis vs. Other Covariance Models\n'
             f'(n={n_true} features, m={m_data} samples, k={k_true} factors)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="468"
     viewBox="0 0 780 468" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="40" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">n &#x226b; m problem</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >full &#x3a3; has O(n&#xb2;) params; singular covariance when n &#x226b; m</text>
  <line x1="102" y1="58" x2="102" y2="120"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="80" font-size="11.5" font-style="italic" fill="#555"
        >introduce k-dim latent z ~ N(0,I)</text>
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Factor model</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">z ~ N(0,I), &#x3b5; ~ N(0,&#x3a8;)</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >x = &#x3bc; + &#x39b;z + &#x3b5; &#x2014; &#x39b; is n&#xd7;k loading matrix, &#x3a8; diagonal noise</text>
  <line x1="102" y1="158" x2="102" y2="220"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="180" font-size="11.5" font-style="italic" fill="#555"
        >compute joint p(x,z) and marginal p(x)</text>
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="235" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Marginal</text>
  <text x="102" y="252" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">x ~ N(&#x3bc;, &#x39b;&#x39b;&#x1d40;+&#x3a8;)</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >covariance = &#x39b;&#x39b;&#x1d40; + &#x3a8; &#x2014; low-rank + diagonal</text>
  <line x1="102" y1="258" x2="102" y2="320"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="280" font-size="11.5" font-style="italic" fill="#555"
        >fit parameters &#x3bc;, &#x39b;, &#x3a8; via EM</text>
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="335" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Fitted FA</text>
  <text x="102" y="352" font-size="12.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">model</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >compact n&#xd7;k representation; EM uses Gaussian conditionals &#x2014; see next notebook</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| Generative model | $z \sim \mathcal{N}(0,I_k)$, $\varepsilon \sim \mathcal{N}(0,\Psi)$, $x = \mu + \Lambda z + \varepsilon$ | $z$ is the $k$-dimensional latent factor |
| Marginal distribution | $x \sim \mathcal{N}(\mu,\; \Lambda\Lambda^T + \Psi)$ | Covariance = low-rank + diagonal |
| Parameters | $\mu \in \mathbb{R}^n$, $\Lambda \in \mathbb{R}^{n \times k}$, $\Psi = \text{diag}(\psi_1,\ldots,\psi_n)$ | $O(nk + n)$ vs $O(n^2)$ for full covariance |
| Motivation | Works when $n \gg m$ | Full covariance matrix is singular; FA is not |
| Fitting | EM algorithm | No closed-form MLE when $z$ is latent |

**Key insight:** factor analysis models the data covariance as $\Lambda\Lambda^T + \Psi$ — a low-rank structure plus diagonal noise; this has $O(nk)$ parameters instead of $O(n^2)$, making it feasible when $n \gg m$.